In [115]:
import pandas as pd

df = pd.read_csv("../data/processed/cleeaned_fraud_train.csv")
df

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,merch_long,is_fraud,date_of_birth,week_day,hour,age,age_group,city_group,distance,distance_group
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,-82.048315,0,1988-03-09,Tuesday,0,31,25-34,small_town,78.597680,metropolitan_area
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,-118.186462,0,1978-06-21,Tuesday,0,41,35-49,rural,30.212218,metropolitan_area
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,-112.154481,0,1962-01-19,Tuesday,0,57,50-64,small_town,108.206235,country
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,-112.561071,0,1967-01-12,Tuesday,0,52,50-64,small_town,95.673366,metropolitan_area
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,-78.632459,0,1986-03-28,Tuesday,0,33,25-34,rural,77.556853,metropolitan_area
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,2020-06-21 12:12:08,30263540414123,fraud_Reichel Inc,entertainment,15.56,Erik,Patterson,M,162 Jessica Row Apt. 072,Hatch,...,-111.690765,0,1961-11-24,Sunday,12,59,50-64,rural,119.752305,country
1296671,2020-06-21 12:12:19,6011149206456997,fraud_Abernathy and Sons,food_dining,51.70,Jeffrey,White,M,8617 Holmes Terrace Suite 651,Tuscarora,...,-78.246528,0,1979-12-11,Sunday,12,41,35-49,rural,75.104191,metropolitan_area
1296672,2020-06-21 12:12:32,3514865930894695,fraud_Stiedemann Ltd,food_dining,105.93,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,High Rolls Mountain Park,...,-105.130529,0,1967-08-30,Sunday,12,53,50-64,rural,99.047874,metropolitan_area
1296673,2020-06-21 12:13:36,2720012583106919,"fraud_Reinger, Weissnat and Strosin",food_dining,74.90,Joseph,Murray,M,42933 Ryan Underpass,Manderson,...,-103.241160,0,1980-08-18,Sunday,12,40,35-49,small_town,84.627772,metropolitan_area


# Fraud KPIs

In [116]:
def get_total_transactions(df:pd.DataFrame):
    return len(df)

def get_total_fraud_transactions(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    return len(df_fraud)

def get_fraud_rate(df):
    total_fraud = get_total_fraud_transactions(df)
    total = get_total_transactions(df)
    return total_fraud/total * 100

def get_total_sum(df:pd.DataFrame):
    return df['amt'].sum()

def get_fraud_loss_sum(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    return df_fraud["amt"].sum()

def get_fraud_loss_rate(df:pd.DataFrame):
    total_sum = get_total_sum(df)
    fraud_sum = get_fraud_loss_sum(df)
    return fraud_sum/total_sum * 100



In [117]:
print(get_total_transactions(df), get_total_fraud_transactions(df))

1296675 7506


In [118]:
print(f"fraud rate is {get_fraud_rate(df):.2}%, total loss rate from fraud is {get_fraud_loss_rate(df):.2}%")

fraud rate is 0.58%, total loss rate from fraud is 4.4%


# Time Analysis

In [119]:
def get_fraud_rate_by_hour(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    fraud_hour_df = df_fraud.groupby(['hour']).count()["first"]

    hour_df = df.groupby(['hour']).count()["first"]

    fraud_rate_by_hour = fraud_hour_df / hour_df * 100
    
    return fraud_rate_by_hour

def get_fraud_rate_by_day(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    fraud_week_day_df = df_fraud.groupby(['week_day']).count()["first"]

    hour_df = df.groupby(['week_day']).count()["first"]

    fraud_rate_by_week_day = fraud_week_day_df / hour_df * 100
    
    return fraud_rate_by_week_day

# 3. Category & Merchant Risk


In [120]:


def get_fraud_risk_rate(df, by):
    grouped = df.groupby(by)["is_fraud"].agg(
        total="count",
        frauds="sum"
    ).reset_index()

    grouped["risk"] = grouped["frauds"] / grouped["total"] * 100

    return grouped.sort_values("risk", ascending=False)



In [121]:
category_fraud_risk_rate_df = get_fraud_risk_rate(df,"category")
category_fraud_risk_rate_df

,category,total,frauds,risk
11,shopping_net,97543,1713,1.756149
8,misc_net,63287,915,1.445795
4,grocery_pos,123638,1743,1.409761
12,shopping_pos,116672,843,0.722538
2,gas_transport,131659,618,0.469394
9,misc_pos,79655,250,0.313853
3,grocery_net,45452,134,0.294817
13,travel,40507,116,0.286370
0,entertainment,94014,233,0.247835
10,personal_care,90758,220,0.242403


# 4. Customer Segments

In [122]:
age_group_fraud_risk_rate_df = get_fraud_risk_rate(df, "age_group")    
age_group_fraud_risk_rate_df

,age_group,total,frauds,risk
5,65+,202312,1504,0.743406
4,50-64,267678,1957,0.731102
1,18-24,97491,660,0.676986
2,25-34,278361,1370,0.492167
3,35-49,437403,1955,0.446956
0,0-17,13430,60,0.446761


In [123]:
city_size_fraud_risk_rate_df = get_fraud_risk_rate(df, "city_group") 
city_size_fraud_risk_rate_df

,city_group,total,frauds,risk
1,medium_city,113165,767,0.677771
0,large_city,32895,215,0.653595
2,mega_city,32814,194,0.591211
3,rural,398055,2299,0.577558
5,small_town,509556,2865,0.562254
4,small_city,210190,1166,0.554736


In [124]:
distance_fraud_risk_rate_df = get_fraud_risk_rate(df, by="distance_group")
distance_fraud_risk_rate_df

,distance_group,total,frauds,risk
3,district,106,1,0.943396
4,metropolitan_area,951432,5536,0.581860
2,country,302853,1747,0.576848
1,city_suburbs,31824,170,0.534188
0,city,10460,52,0.497132


In [127]:
grouped_amt = df.groupby(["cc_num"])["amt"].agg(
    avg="mean",
    max="max"
)
grouped_amt

,avg,max
cc_num,,
60416207185,56.023366,3075.09
60422928733,69.000784,1290.37
60423098130,115.046333,27119.77
60427851591,111.987898,1164.36
60487002085,50.726028,750.39
...,...,...
4958589671582726883,66.377839,4292.86
4973530368125489546,78.373288,8749.44
4980323467523543940,74.436429,1327.43
